<a href="https://colab.research.google.com/github/anuragpandey4/dl/blob/main/buildingLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# setup

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import time
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device : " ,device)
if device == "cuda":
  print("GPU : ", torch.cuda.get_device_name(0))

Device :  cuda
GPU :  Tesla T4


# data loading

In [5]:
import urllib.request
from pathlib import Path

# Define constants
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FILE_PATH = Path("shakespeare.txt")

# 1. Download the file only if it isn't already saved locally
if not FILE_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, FILE_PATH)

# 2. Read the text with explicit utf-8 encoding
with open(FILE_PATH, "r", encoding="utf-8") as f:
    text = f.read()

# 3. Print the results
print(f"Total characters: {len(text):,}")
print(f"First 200 characters:\n{text[:200]}")

Total characters: 1,115,394
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


# character level tokenization

In [8]:
# build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [10]:
# char to int and vice versa mapping
char_to_idx = {ch:i for i,ch in enumerate(chars)}
idx_to_char = {i:ch for i,ch in enumerate(chars)}

#encoder and decoder
def encode(s):
  return [char_to_idx[ch] for ch in s]

def decode(ids):
  return "".join([idx_to_char[idx] for idx in ids])

In [11]:
print(f"vocab size : {vocab_size}")
print(f"Characters : {''.join(chars)}")
print(f"Encoding <-> Decoding")
print(f"'Anurag' --> {encode('Anurag')}")
print(f"{encode('Anurag')} --> {decode(encode('Anurag'))}")

vocab size : 65
Characters : 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Encoding <-> Decoding
'Anurag' --> [13, 52, 59, 56, 39, 45]
[13, 52, 59, 56, 39, 45] --> Anurag
